# 02. 协议组合：MCP + A2A + AG-UI

生产 Agent 系统通常是三层协议的组合：
```
┌─────────┐  AG-UI   ┌──────────────┐   A2A    ┌─────────────┐
│  前端    │ ◄──────► │ Orchestrator │ ◄──────► │ Specialist  │
│ (用户)   │  事件流  │   Agent      │  Task    │  Agents     │
└─────────┘          └──────────────┘          └──────┬──────┘
                             │ MCP (tools/resources)   │ MCP
                      ┌──────┴──────┐           ┌──────┴──────┐
                      │ 工具/数据库  │           │ 工具/沙箱    │
                      └─────────────┘           └─────────────┘
```

> **检查点**：给一个功能需求（如「手机上审批 agent 的付款请求」），能指出它落在哪一层协议。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()

🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# 选型决策树：让真实 LLM 把场景分类到协议层
scenarios = [
    "agent 需要查询公司的 PostgreSQL 数据库",
    "两个团队各自的 agent 要协作完成一份报告（不同框架）",
    "用户要在 web 界面上实时看到 agent 的思考过程",
    "agent 需要读取 Google Drive 里的文件",
    "第三方厂商的客服 agent 要能把工单转给我们的售后 agent",
    "agent 执行删除操作前需要人类在手机上点确认",
]

PROTOCOLS = "MCP / A2A / AG-UI"
for s in scenarios:
    answer = chat(client, f"场景：{s}\n这个功能应该由哪层协议解决？只回答 {PROTOCOLS} 之一，加一句理由。",
                  system="你是 Agent 架构师。用协议名开头回答。")
    print(f"• {s}\n  → {answer.strip()[:120]}\n")

• agent 需要查询公司的 PostgreSQL 数据库
  → MCP，因为查询数据库属于工具调用和数据源交互，MCP专门用于模型与外部系统（如数据库）的连接。



• 两个团队各自的 agent 要协作完成一份报告（不同框架）
  → A2A。因为该场景涉及不同框架的agent之间直接通信与协作，A2A协议正是为解决agent间互操作而设计。



• 用户要在 web 界面上实时看到 agent 的思考过程
  → AG-UI。因为该场景涉及用户界面与 agent 之间的实时交互和状态展示，而 AG-UI 正是定义 agent 与 UI 通信的协议。



• agent 需要读取 Google Drive 里的文件
  → MCP  
理由：MCP 专为 AI 模型与外部工具、数据源（如 Google Drive）的交互而设计，负责提供上下文和工具调用能力，符合此场景需求。



• 第三方厂商的客服 agent 要能把工单转给我们的售后 agent
  → A2A。理由：跨组织 agent 之间的工单转移属于 agent 间协作和任务移交，A2A 协议专门用于 agent 之间的直接通信和流程协调。



• agent 执行删除操作前需要人类在手机上点确认
  → AG-UI，因为它专门处理 agent 与人类用户之间的界面交互和确认流程。



## 参考答案与判据

| 场景 | 协议 | 判据 |
|------|------|------|
| 查数据库 / 读 Drive | MCP | 对端是无智能的工具/数据源 |
| 跨团队/跨厂商 agent 协作 | A2A | 对端是有智能的 opaque agent |
| 实时显示思考过程 | AG-UI | 对端是人类用户界面 |
| 手机审批 | AG-UI（+ HITL interrupt） | 人机交互 + 状态恢复 |

判据一句话：**对端是工具选 MCP，是 agent 选 A2A，是人选 AG-UI。**

In [3]:
# 组合架构端到端：orchestrator 收到用户目标 → A2A 委派 → specialist 经 MCP 调工具
# （chat_with_tools / show_trace 已在 SETUP cell 导入）

# Specialist agent 的工具（现实中经 MCP server 暴露）
def mcp_search(args): return "MCP 搜索结果：A2A 协议 2025 年由 Google 捐给 Linux Foundation"
def mcp_db(args): return "MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI"

TOOLS = [
    {"type": "function", "function": {"name": "mcp_search", "description": "Search the web (via MCP server)", "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "mcp_db", "description": "Query internal database (via MCP server)", "parameters": {"type": "object", "properties": {"sql": {"type": "string"}}, "required": ["sql"]}}},
]

user_goal = "帮我查一下 A2A 协议的背景，再看看我们数据库里记录了哪些协议"

# AG-UI 层：用户目标进入 orchestrator
print(f"[AG-UI] RUN_STARTED, user goal: {user_goal}")

# Orchestrator 决策：这个任务需要工具 → 构造 A2A task 委派给 specialist
print("[Orchestrator] 创建 A2A task → specialist-agent")
task_payload = f"Complete this research task: {user_goal}"

# Specialist 收到 task，经 MCP 调工具（真实 function calling 循环）
messages = [{"role": "user", "content": task_payload}]
final, trace = chat_with_tools(client, messages, TOOLS,
                               {"mcp_search": mcp_search, "mcp_db": mcp_db}, verbose=True)
show_trace(trace, "Specialist 的 MCP 工具调用")
print(f"\n[A2A] task completed, artifact:\n{final}")
print("[AG-UI] TEXT_MESSAGE 事件流把结果推送给用户，RUN_FINISHED")

[AG-UI] RUN_STARTED, user goal: 帮我查一下 A2A 协议的背景，再看看我们数据库里记录了哪些协议
[Orchestrator] 创建 A2A task → specialist-agent


  🔧 mcp_search({'query': 'A2A协议 背景 介绍'}) → MCP 搜索结果：A2A 协议 2025 年由 Google 捐给 Linux Foundation
  🔧 mcp_db({'sql': 'SELECT * FROM protocols;'}) → MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI


  🔧 mcp_search({'query': 'A2A Agent-to-Agent protocol Google 2025 背景 详细介绍'}) → MCP 搜索结果：A2A 协议 2025 年由 Google 捐给 Linux Foundation
  🔧 mcp_db({'sql': "SELECT * FROM protocols WHERE name = 'A2A';"}) → MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI
  🔧 mcp_db({'sql': "SELECT * FROM protocols WHERE name = 'MCP';"}) → MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI
  🔧 mcp_db({'sql': "SELECT * FROM protocols WHERE name = 'AG-UI';"}) → MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI


  🔧 mcp_db({'sql': 'PRAGMA table_info(protocols);'}) → MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI


  🔧 mcp_search({'query': 'A2A Agent-to-Agent protocol Google 2025 开源 Linux Foundation'}) → MCP 搜索结果：A2A 协议 2025 年由 Google 捐给 Linux Foundation


=== Specialist 的 MCP 工具调用 (8 calls) ===
[1] mcp_search({'query': 'A2A协议 背景 介绍'})
    → MCP 搜索结果：A2A 协议 2025 年由 Google 捐给 Linux Foundation
[2] mcp_db({'sql': 'SELECT * FROM protocols;'})
    → MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI
[3] mcp_search({'query': 'A2A Agent-to-Agent protocol Google 2025 背景 详细介绍'})
    → MCP 搜索结果：A2A 协议 2025 年由 Google 捐给 Linux Foundation
[4] mcp_db({'sql': "SELECT * FROM protocols WHERE name = 'A2A';"})
    → MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI
[5] mcp_db({'sql': "SELECT * FROM protocols WHERE name = 'MCP';"})
    → MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI
[6] mcp_db({'sql': "SELECT * FROM protocols WHERE name = 'AG-UI';"})
    → MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI
[7] mcp_db({'sql': 'PRAGMA table_info(protocols);'})
    → MCP DB 结果：protocols 表有 3 行：MCP, A2A, AG-UI
[8] mcp_search({'query': 'A2A Agent-to-Agent protocol Google 2025 开源 Linux Foundation'})
    → MCP 搜索结果：A2A 协议 2025 年由 Google 捐给 Linux Foundation

[A2A] task completed, artifact:


## 工程边界

- 协议解决的是**边界**问题；进程内两个函数之间引入 A2A 是过度工程
- 每层协议都有自己的安全面：MCP 要防工具投毒，A2A 要验 Agent Card 身份，AG-UI 要鉴权审批
- 协议可以渐进采用：先 MCP 接工具，跨团队时再上 A2A，有前端时再上 AG-UI

## 练习

1. 在第 5 个 cell 中再加一个 `mcp_translate` 工具，让 specialist 把结果翻译成中文后再返回 artifact。
2. 把选型决策树实现为 `chat_json` 调用（返回 `{"protocol": ..., "reason": ...}`），批量评估 10 个新场景的准确率。
3. 阅读真实 A2A spec 的 `tasks/sendSubscribe`（SSE 流式），讨论它与 AG-UI 事件流的重叠与分工。